In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

TICKER = "QLD"
SMA_LONG = 250
SMA_MEDIUM = 100
BUFFER = 0.05
VOL_LOOKBACK = 21
VOL_THRESHOLD = 0.40
AR1_LOOKBACK = 30


def find_env_candidates() -> list[Path]:
    seen = set()
    candidates = []

    # Probe cwd and every ancestor, which handles notebooks launched from nested folders.
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / ".env"
        key = str(env_path.resolve())
        if key not in seen:
            seen.add(key)
            candidates.append(env_path)

    # Optional explicit location if user sets it.
    explicit = os.getenv("ENV_FILE_PATH", "").strip()
    if explicit:
        explicit_path = Path(explicit)
        key = str(explicit_path.resolve())
        if key not in seen:
            seen.add(key)
            candidates.append(explicit_path)

    return candidates


def load_env_var_from_file(var_name: str) -> str:
    for env_path in find_env_candidates():
        if not env_path.exists():
            continue

        try:
            for raw_line in env_path.read_text(encoding="utf-8").splitlines():
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue

                if line.startswith("export "):
                    line = line[len("export "):].strip()

                key, value = line.split("=", 1)
                if key.strip() == var_name:
                    value = value.strip().strip('"').strip("'")
                    if value:
                        os.environ[var_name] = value
                        return value
        except Exception:
            continue

    return os.getenv(var_name, "").strip()


def fetch_prices(ticker: str, period: str = "3y") -> pd.DataFrame:
    data = yf.download(ticker, period=period, auto_adjust=True, progress=False)
    if data.empty:
        raise ValueError(f"No price data returned for {ticker}")
    if isinstance(data.columns, pd.MultiIndex):
        column_levels = data.columns.levels
        if "Close" in data.columns.get_level_values(0):
            data.columns = data.columns.get_level_values(0)
        elif len(column_levels) > 1 and "Close" in data.columns.get_level_values(1):
            data.columns = data.columns.get_level_values(1)
        else:
            data.columns = ["_".join(map(str, column)).strip() for column in data.columns]
    if "Close" not in data.columns:
        raise ValueError("Expected a Close column in downloaded data")
    return data.dropna(subset=["Close"]).copy()


def compute_ar1(values: pd.Series) -> float:
    x = values.dropna().to_numpy(dtype=float)
    if len(x) < 3:
        return np.nan
    lagged = x[:-1]
    current = x[1:]
    variance = np.var(lagged)
    if variance == 0:
        return np.nan
    return float(np.cov(lagged, current, ddof=0)[0, 1] / variance)


def make_signals(prices: pd.DataFrame) -> pd.DataFrame:
    close = prices["Close"]
    sma_250 = close.rolling(SMA_LONG).mean()
    sma_100 = close.rolling(SMA_MEDIUM).mean()
    returns = close.pct_change()
    realized_vol = returns.rolling(VOL_LOOKBACK).std() * np.sqrt(252)
    ar1 = returns.rolling(AR1_LOOKBACK).apply(compute_ar1, raw=False)

    signal_1_on = close > sma_250 * (1 + BUFFER)
    signal_1_off = close < sma_250 * (1 - BUFFER)
    signal_1 = np.where(signal_1_on, True, np.where(signal_1_off, False, np.nan))
    signal_1 = pd.Series(signal_1, index=prices.index).ffill().fillna(False)

    signal_2_on = close > sma_100 * (1 + BUFFER)
    signal_2_off = close < sma_100 * (1 - BUFFER)
    signal_2 = np.where(signal_2_on, True, np.where(signal_2_off, False, np.nan))
    signal_2 = pd.Series(signal_2, index=prices.index).ffill().fillna(False)

    signal_3 = realized_vol < VOL_THRESHOLD
    signal_4 = ar1 > 0

    result = pd.DataFrame(
        {
            "Close": close,
            "SMA_250": sma_250,
            "SMA_100": sma_100,
            "21D_Ann_Vol": realized_vol,
            "30D_AR1": ar1,
            "Signal_1_LongTrend": signal_1,
            "Signal_2_MediumTrend": signal_2,
            "Signal_3_LowVol": signal_3,
            "Signal_4_PositiveAR1": signal_4,
        }
    )
    result["Green_Count"] = result[
        ["Signal_1_LongTrend", "Signal_2_MediumTrend", "Signal_3_LowVol", "Signal_4_PositiveAR1"]
    ].sum(axis=1)
    result["State"] = np.where(result["Green_Count"] >= 2, "ON", "OFF")
    return result


def fetch_fred_series(series_id: str, start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.Series:
    api_key = os.getenv("FRED_API_KEY", "").strip()

    # Preferred path: official FRED API when an API key is available.
    if api_key:
        try:
            response = requests.get(
                "https://api.stlouisfed.org/fred/series/observations",
                params={
                    "series_id": series_id,
                    "api_key": api_key,
                    "file_type": "json",
                    "observation_start": start_date.strftime("%Y-%m-%d"),
                    "observation_end": end_date.strftime("%Y-%m-%d"),
                },
                timeout=15,
            )
            response.raise_for_status()
            observations = response.json().get("observations", [])
            df = pd.DataFrame(observations)
            if not df.empty:
                df["date"] = pd.to_datetime(df["date"])
                df["value"] = pd.to_numeric(df["value"].replace(".", np.nan), errors="coerce")
                out = df.set_index("date")["value"].dropna()
                out.name = series_id
                return out
        except Exception:
            pass

    # Fallback path: FRED published CSV endpoint (no key requirement).
    try:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        df = pd.read_csv(url)
        if "DATE" in df.columns and series_id in df.columns:
            df["DATE"] = pd.to_datetime(df["DATE"])
            df = df[(df["DATE"] >= start_date) & (df["DATE"] <= end_date)]
            values = pd.to_numeric(df[series_id], errors="coerce")
            out = pd.Series(values.values, index=df["DATE"], name=series_id).dropna()
            return out
    except Exception:
        pass

    return pd.Series(dtype=float, name=series_id)


def series_last(series: pd.Series) -> float:
    return float(series.dropna().iloc[-1]) if not series.dropna().empty else np.nan


def series_change(series: pd.Series, lookback: int) -> float:
    clean = series.dropna()
    if len(clean) <= lookback:
        return np.nan
    return float(clean.iloc[-1] - clean.iloc[-1 - lookback])


def annualized_3m_change(series: pd.Series) -> pd.Series:
    return ((series / series.shift(3)) ** (4 / 3) - 1) * 100


def series_as_of(series: pd.Series) -> pd.Timestamp:
    clean = series.dropna()
    return clean.index.max() if not clean.empty else pd.NaT


def fmt_date(value: pd.Timestamp) -> str:
    if pd.isna(value):
        return "N/A"
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def bool_text(flag: bool) -> str:
    return "Green" if flag else "Red"


def crash_bool_text(flag: bool) -> str:
    return "Red" if flag else "Green"


def color_status(value: str) -> str:
    text = str(value).strip().upper()
    if text == "GREEN" or text == "ON":
        return "background-color: #d1fae5; color: #065f46; font-weight: 600;"
    if text == "RED" or text == "OFF":
        return "background-color: #fee2e2; color: #991b1b; font-weight: 600;"
    return ""


def color_status_crash(value: str) -> str:
    text = str(value).strip().upper()
    if text == "OFF":
        return "background-color: #d1fae5; color: #065f46; font-weight: 600;"
    if text == "ON":
        return "background-color: #fee2e2; color: #991b1b; font-weight: 600;"
    return color_status(value)


fred_api_key = load_env_var_from_file("FRED_API_KEY")
if fred_api_key:
    print("FRED_API_KEY loaded from environment or .env")
else:
    print("FRED_API_KEY not found. FRED fields may show N/A")

prices = fetch_prices(TICKER)
signals = make_signals(prices)
latest = signals.dropna(subset=["SMA_250", "SMA_100", "21D_Ann_Vol", "30D_AR1"]).iloc[-1]
latest_date = pd.Timestamp(latest.name)

summary = pd.DataFrame(
    {
        "Metric": [
            "Signal 1 - Long trend",
            "Signal 2 - Medium trend",
            "Signal 3 - Realized volatility",
            "Signal 4 - Return persistence AR(1)",
            "Green count",
            "State",
        ],
        "Current": [
            f"Close {latest['Close']:.2f} vs SMA250 {latest['SMA_250']:.2f}",
            f"Close {latest['Close']:.2f} vs SMA100 {latest['SMA_100']:.2f}",
            f"21D ann vol {latest['21D_Ann_Vol']:.2%}",
            f"AR(1) {latest['30D_AR1']:.4f}",
            f"{int(latest['Green_Count'])}/4",
            latest['State'],
        ],
        "Status": [
            bool_text(bool(latest['Signal_1_LongTrend'])),
            bool_text(bool(latest['Signal_2_MediumTrend'])),
            bool_text(bool(latest['Signal_3_LowVol'])),
            bool_text(bool(latest['Signal_4_PositiveAR1'])),
            "",
            latest['State'],
        ],
        "Data_Date": [fmt_date(latest_date)] * 6,
    }
)

print(f"Green: {int(latest['Green_Count'])}, Red: {4 - int(latest['Green_Count'])}, State: {latest['State']}")
display(summary.style.map(color_status, subset=["Status"]))


# Macro regime dashboard (FRED + yfinance)
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10)

fred = {
    "fed_funds": fetch_fred_series("FEDFUNDS", start_date, end_date),
    "dgs2": fetch_fred_series("DGS2", start_date, end_date),
    "napm": fetch_fred_series("NAPM", start_date, end_date),
    "dgs10": fetch_fred_series("DGS10", start_date, end_date),
    "t10yie": fetch_fred_series("T10YIE", start_date, end_date),
    "unrate": fetch_fred_series("UNRATE", start_date, end_date),
    "cpi": fetch_fred_series("CPIAUCSL", start_date, end_date),
    "hy_oas": fetch_fred_series("BAMLH0A0HYM2", start_date, end_date),
    "curve": fetch_fred_series("T10Y2Y", start_date, end_date),
    "indpro": fetch_fred_series("INDPRO", start_date, end_date),
}

vix_df = yf.download("^VIX", period="1y", auto_adjust=True, progress=False)
if isinstance(vix_df.columns, pd.MultiIndex):
    if "Close" in vix_df.columns.get_level_values(0):
        vix_df.columns = vix_df.columns.get_level_values(0)
    elif "Close" in vix_df.columns.get_level_values(1):
        vix_df.columns = vix_df.columns.get_level_values(1)
vix = vix_df["Close"].dropna() if "Close" in vix_df.columns else pd.Series(dtype=float)

real_10y = fred["dgs10"] - fred["t10yie"]
cpi_yoy = fred["cpi"].pct_change(12) * 100

fed_funds_6m = series_change(fred["fed_funds"], 6)
dgs2_3m = series_change(fred["dgs2"], 63)
indpro_3m_ann = annualized_3m_change(fred["indpro"])
indpro_3m_ann_now = series_last(indpro_3m_ann)
indpro_3m_ann_prev = float(indpro_3m_ann.dropna().iloc[-2]) if len(indpro_3m_ann.dropna()) >= 2 else np.nan
indpro_3m_ann_neg_2 = bool(len(indpro_3m_ann.dropna()) >= 2 and indpro_3m_ann.dropna().iloc[-1] < 0 and indpro_3m_ann.dropna().iloc[-2] < 0)
napm_now = series_last(fred["napm"])
napm_6m = series_change(fred["napm"], 6)
real10_now = series_last(real_10y)
real10_3m = series_change(real_10y, 63)
unrate_6m = series_change(fred["unrate"], 6)
cpi_yoy_now = series_last(cpi_yoy)
hy_now = series_last(fred["hy_oas"])
hy_3m = series_change(fred["hy_oas"], 63)
curve_now = series_last(fred["curve"])
curve_min_18m = float(fred["curve"].dropna().tail(378).min()) if not fred["curve"].dropna().empty else np.nan
vix_now = series_last(vix)
vix_min_10d = float(vix.tail(10).min()) if not vix.empty else np.nan
vix_jump_10d = vix_now - vix_min_10d if pd.notna(vix_now) and pd.notna(vix_min_10d) else np.nan

regime1_conditions = {
    "2Y rising rapidly": pd.notna(dgs2_3m) and dgs2_3m >= 0.40,
    "Fed hiking": pd.notna(fed_funds_6m) and fed_funds_6m > 0,
    "Positive real 10Y": pd.notna(real10_now) and real10_now > 0,
    "INDPRO > 0 and rising": pd.notna(indpro_3m_ann_now) and pd.notna(indpro_3m_ann_prev) and indpro_3m_ann_now > 0 and indpro_3m_ann_now >= indpro_3m_ann_prev,
}

regime2_conditions = {
    "Real 10Y negative/falling": (pd.notna(real10_now) and real10_now < 0) or (pd.notna(real10_3m) and real10_3m <= -0.50),
    "Unemployment rising": pd.notna(unrate_6m) and unrate_6m >= 0.30,
    "Inflation elevated": pd.notna(cpi_yoy_now) and cpi_yoy_now >= 3.0,
}

regime3_conditions = {
    "HY OAS > 500 bps": pd.notna(hy_now) and hy_now >= 5.0,
    "HY OAS widening fast": pd.notna(hy_3m) and hy_3m >= 1.0,
    "Prior inversion occurred": pd.notna(curve_min_18m) and curve_min_18m <= -0.50,
    "Recession now materializing": pd.notna(unrate_6m) and unrate_6m >= 0.30,
}

regime4_conditions = {
    "VIX > 35": pd.notna(vix_now) and vix_now > 35,
    "VIX jump in 5-10d": pd.notna(vix_jump_10d) and vix_jump_10d >= 15,
    "HY OAS widening fast": pd.notna(hy_3m) and hy_3m >= 0.75,
    "No curve stress shift": pd.notna(curve_now) and curve_now > -1.25,
}

regime1_on = sum(regime1_conditions.values()) >= 3
regime2_on = sum(regime2_conditions.values()) >= 2
regime3_on = sum(regime3_conditions.values()) >= 3
regime4_on = sum(regime4_conditions.values()) >= 3

macro_table = pd.DataFrame(
    [
        ["Regime 1 - Liquidity Rate-Hike", "Fed Funds 6M change", f"{fed_funds_6m:.2f} pp" if pd.notna(fed_funds_6m) else "N/A", "> 0", crash_bool_text(regime1_conditions["Fed hiking"]), fmt_date(series_as_of(fred["fed_funds"]))],
        ["Regime 1 - Liquidity Rate-Hike", "2Y Treasury 3M change", f"{dgs2_3m:.2f} pp" if pd.notna(dgs2_3m) else "N/A", ">= 0.40 pp", crash_bool_text(regime1_conditions["2Y rising rapidly"]), fmt_date(series_as_of(fred["dgs2"]))],
        ["Regime 1 - Liquidity Rate-Hike", "Real 10Y level", f"{real10_now:.2f}%" if pd.notna(real10_now) else "N/A", "> 0%", crash_bool_text(regime1_conditions["Positive real 10Y"]), fmt_date(series_as_of(real_10y))],
        ["Regime 1 - Liquidity Rate-Hike", "INDPRO 3M annualized change", f"{indpro_3m_ann_now:.2%}, prev={indpro_3m_ann_prev:.2%}, neg2={indpro_3m_ann_neg_2}" if pd.notna(indpro_3m_ann_now) and pd.notna(indpro_3m_ann_prev) else "N/A", "> 0% and rising", crash_bool_text(regime1_conditions["INDPRO > 0 and rising"]), fmt_date(series_as_of(fred["indpro"]))],
        ["Regime 2 - Stagflationary", "Real 10Y level/change", f"{real10_now:.2f}%, d3m={real10_3m:.2f} pp" if pd.notna(real10_now) and pd.notna(real10_3m) else "N/A", "<0 or d3m<=-0.50 pp", crash_bool_text(regime2_conditions["Real 10Y negative/falling"]), fmt_date(series_as_of(real_10y))],
        ["Regime 2 - Stagflationary", "Unemployment 6M change", f"{unrate_6m:.2f} pp" if pd.notna(unrate_6m) else "N/A", ">= 0.30 pp", crash_bool_text(regime2_conditions["Unemployment rising"]), fmt_date(series_as_of(fred["unrate"]))],
        ["Regime 2 - Stagflationary", "CPI YoY", f"{cpi_yoy_now:.2f}%" if pd.notna(cpi_yoy_now) else "N/A", ">= 3%", crash_bool_text(regime2_conditions["Inflation elevated"]), fmt_date(series_as_of(cpi_yoy))],
        ["Regime 3 - Structural Systemic", "HY OAS level", f"{hy_now:.2f}%" if pd.notna(hy_now) else "N/A", ">= 5.0%", crash_bool_text(regime3_conditions["HY OAS > 500 bps"]), fmt_date(series_as_of(fred["hy_oas"]))],
        ["Regime 3 - Structural Systemic", "HY OAS 3M change", f"{hy_3m:.2f} pp" if pd.notna(hy_3m) else "N/A", ">= 1.0 pp", crash_bool_text(regime3_conditions["HY OAS widening fast"]), fmt_date(series_as_of(fred["hy_oas"]))],
        ["Regime 3 - Structural Systemic", "10Y-2Y min over 18M", f"{curve_min_18m:.2f} pp" if pd.notna(curve_min_18m) else "N/A", "<= -0.50 pp", crash_bool_text(regime3_conditions["Prior inversion occurred"]), fmt_date(series_as_of(fred["curve"]))],
        ["Regime 3 - Structural Systemic", "Unemployment 6M change", f"{unrate_6m:.2f} pp" if pd.notna(unrate_6m) else "N/A", ">= 0.30 pp", crash_bool_text(regime3_conditions["Recession now materializing"]), fmt_date(series_as_of(fred["unrate"]))],
        ["Regime 4 - Exogenous Flash", "VIX absolute", f"{vix_now:.2f}" if pd.notna(vix_now) else "N/A", "> 35", crash_bool_text(regime4_conditions["VIX > 35"]), fmt_date(series_as_of(vix))],
        ["Regime 4 - Exogenous Flash", "VIX 10D jump", f"{vix_jump_10d:.2f}" if pd.notna(vix_jump_10d) else "N/A", ">= 15", crash_bool_text(regime4_conditions["VIX jump in 5-10d"]), fmt_date(series_as_of(vix))],
        ["Regime 4 - Exogenous Flash", "HY OAS 3M change", f"{hy_3m:.2f} pp" if pd.notna(hy_3m) else "N/A", ">= 0.75 pp", crash_bool_text(regime4_conditions["HY OAS widening fast"]), fmt_date(series_as_of(fred["hy_oas"]))],
        ["Regime 4 - Exogenous Flash", "10Y-2Y current", f"{curve_now:.2f} pp" if pd.notna(curve_now) else "N/A", "> -1.25 pp", crash_bool_text(regime4_conditions["No curve stress shift"]), fmt_date(series_as_of(fred["curve"]))],
    ],
    columns=["Regime", "Indicator", "Current", "Rule", "Status", "Data_Date"],
)

regime_state = pd.DataFrame(
    {
        "Regime": [
            "Regime 1 - Liquidity Rate-Hike",
            "Regime 2 - Stagflationary",
            "Regime 3 - Structural Systemic",
            "Regime 4 - Exogenous Flash",
        ],
        "Green_Count": [
            sum(regime1_conditions.values()),
            sum(regime2_conditions.values()),
            sum(regime3_conditions.values()),
            sum(regime4_conditions.values()),
        ],
        "Needed": [3, 2, 3, 3],
        "State": [
            "ON" if regime1_on else "OFF",
            "ON" if regime2_on else "OFF",
            "ON" if regime3_on else "OFF",
            "ON" if regime4_on else "OFF",
        ],
        "Data_Date": [
            fmt_date(max(series_as_of(fred["fed_funds"]), series_as_of(fred["dgs2"]), series_as_of(real_10y), series_as_of(fred["indpro"]))),
            fmt_date(max(series_as_of(real_10y), series_as_of(fred["unrate"]), series_as_of(cpi_yoy))),
            fmt_date(max(series_as_of(fred["hy_oas"]), series_as_of(fred["curve"]), series_as_of(fred["unrate"]))),
            fmt_date(max(series_as_of(vix), series_as_of(fred["hy_oas"]), series_as_of(fred["curve"]))),
        ],
    }
)

print("\nMacro regime tracker:")
print(
    f"R1={sum(regime1_conditions.values())}/4 ({'ON' if regime1_on else 'OFF'}), "
    f"R2={sum(regime2_conditions.values())}/3 ({'ON' if regime2_on else 'OFF'}), "
    f"R3={sum(regime3_conditions.values())}/4 ({'ON' if regime3_on else 'OFF'}), "
    f"R4={sum(regime4_conditions.values())}/4 ({'ON' if regime4_on else 'OFF'})"
)

display(macro_table.style.map(color_status, subset=["Status"]))
display(regime_state.style.map(color_status_crash, subset=["State"]))

FRED_API_KEY loaded from environment or .env
Green: 2, Red: 2, State: ON


,Metric,Current,Status,Data_Date
0,Signal 1 - Long trend,Close 82.50 vs SMA250 73.94,Green,2026-07-27
1,Signal 2 - Medium trend,Close 82.50 vs SMA100 82.06,Green,2026-07-27
2,Signal 3 - Realized volatility,21D ann vol 47.09%,Red,2026-07-27
3,Signal 4 - Return persistence AR(1),AR(1) -0.0381,Red,2026-07-27
4,Green count,2/4,,2026-07-27
5,State,ON,ON,2026-07-27



Macro regime tracker:
R1=3/4 (ON), R2=1/3 (OFF), R3=0/4 (OFF), R4=1/4 (OFF)


,Regime,Indicator,Current,Rule,Status,Data_Date
0,Regime 1 - Liquidity Rate-Hike,Fed Funds 6M change,-0.09 pp,> 0,Green,2026-06-01
1,Regime 1 - Liquidity Rate-Hike,2Y Treasury 3M change,0.50 pp,>= 0.40 pp,Red,2026-07-24
2,Regime 1 - Liquidity Rate-Hike,Real 10Y level,2.43%,> 0%,Red,2026-07-24
3,Regime 1 - Liquidity Rate-Hike,INDPRO 3M annualized change,"134.36%, prev=83.06%, neg2=False",> 0% and rising,Red,2026-06-01
4,Regime 2 - Stagflationary,Real 10Y level/change,"2.43%, d3m=0.51 pp",<0 or d3m<=-0.50 pp,Green,2026-07-24
5,Regime 2 - Stagflationary,Unemployment 6M change,-0.20 pp,>= 0.30 pp,Green,2026-06-01
6,Regime 2 - Stagflationary,CPI YoY,3.73%,>= 3%,Red,2026-06-01
7,Regime 3 - Structural Systemic,HY OAS level,2.79%,>= 5.0%,Green,2026-07-24
8,Regime 3 - Structural Systemic,HY OAS 3M change,-0.03 pp,>= 1.0 pp,Green,2026-07-24
9,Regime 3 - Structural Systemic,10Y-2Y min over 18M,0.20 pp,<= -0.50 pp,Green,2026-07-24


,Regime,Green_Count,Needed,State,Data_Date
0,Regime 1 - Liquidity Rate-Hike,3,3,ON,2026-07-24
1,Regime 2 - Stagflationary,1,2,OFF,2026-07-24
2,Regime 3 - Structural Systemic,0,3,OFF,2026-07-24
3,Regime 4 - Exogenous Flash,1,3,OFF,2026-07-27


In [8]:
HIST_TICKER = "QQQ"
HIST_DAILY_LEVERAGE = 1.0
HIST_START_DATE = "2000-01-01"
HIST_END_DATE = None
HIST_ONLY_INFLECTION_FLIPS = True


def _download_price_window(ticker: str, start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    data = yf.download(
        ticker,
        start=start_date.strftime("%Y-%m-%d"),
        end=(end_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False,
    )
    if data.empty:
        raise ValueError(f"No price data returned for {ticker}")
    if isinstance(data.columns, pd.MultiIndex):
        if "Close" in data.columns.get_level_values(0):
            data.columns = data.columns.get_level_values(0)
        elif "Close" in data.columns.get_level_values(1):
            data.columns = data.columns.get_level_values(1)
        else:
            data.columns = ["_".join(map(str, column)).strip() for column in data.columns]
    if "Close" not in data.columns:
        raise ValueError("Expected a Close column in downloaded data")
    return data.dropna(subset=["Close"]).copy()



def _apply_daily_leverage(prices: pd.DataFrame, daily_leverage: float) -> pd.DataFrame:
    if daily_leverage <= 0:
        raise ValueError("daily_leverage must be greater than 0")

    if daily_leverage == 1:
        return prices.copy()

    leveraged = prices.copy()
    close = leveraged["Close"].astype(float)
    leveraged_returns = close.pct_change().fillna(0.0).mul(daily_leverage)
    leveraged["Close"] = close.iloc[0] * (1 + leveraged_returns).cumprod()
    return leveraged



def _state_text(flag: bool) -> str:
    return "ON" if bool(flag) else "OFF"



def historical_signal_transitions(
    ticker: str = HIST_TICKER,
    daily_leverage: float = HIST_DAILY_LEVERAGE,
    start_date=HIST_START_DATE,
    end_date=HIST_END_DATE,
    only_inflection_flips: bool = HIST_ONLY_INFLECTION_FLIPS,
) -> pd.DataFrame:
    end_ts = pd.Timestamp.today().normalize() if end_date in (None, "") else pd.Timestamp(end_date).normalize()
    start_ts = end_ts - pd.DateOffset(years=3) if start_date in (None, "") else pd.Timestamp(start_date).normalize()

    if start_ts > end_ts:
        raise ValueError("start_date must be on or before end_date")

    # Fetch extra history so the rolling indicators are valid at the requested start date.
    fetch_start = start_ts - pd.DateOffset(months=18)
    raw_prices = _download_price_window(ticker, fetch_start, end_ts)
    analyzed_prices = _apply_daily_leverage(raw_prices, float(daily_leverage))
    signals = make_signals(analyzed_prices)

    required_cols = ["SMA_250", "SMA_100", "21D_Ann_Vol", "30D_AR1"]
    history = signals.loc[start_ts:end_ts].copy()
    history = history.dropna(subset=required_cols)
    if history.empty:
        raise ValueError("No usable observations in the requested window. Expand the date range or start earlier.")

    signal_cols = [
        "Signal_1_LongTrend",
        "Signal_2_MediumTrend",
        "Signal_3_LowVol",
        "Signal_4_PositiveAR1",
    ]
    state_band = history["Green_Count"] >= 2

    if only_inflection_flips:
        change_mask = state_band.ne(state_band.shift())
        change_mask.iloc[0] = False
    else:
        change_mask = history[signal_cols + ["State"]].ne(history[signal_cols + ["State"]].shift()).any(axis=1)
        change_mask.iloc[0] = True

    transitions = history.loc[change_mask].copy()
    changed_signals = history[signal_cols + ["State"]].ne(history[signal_cols + ["State"]].shift())
    transitions["Changed_Signals"] = changed_signals.loc[change_mask].apply(
        lambda row: ", ".join([name for name, changed in row.items() if changed]) if row.any() else "",
        axis=1,
    )
    if not only_inflection_flips and not transitions.empty:
        transitions.loc[transitions.index[0], "Changed_Signals"] = "Initial state"

    transitions = transitions.rename(columns={"Close": "Analyzed_Close", "State": "Composite_State", "Green_Count": "Signal_Count"})
    transitions["Ticker"] = ticker
    transitions["Daily_Leverage"] = float(daily_leverage)
    transitions["Signal_1_LongTrend"] = transitions["Signal_1_LongTrend"].map(_state_text)
    transitions["Signal_2_MediumTrend"] = transitions["Signal_2_MediumTrend"].map(_state_text)
    transitions["Signal_3_LowVol"] = transitions["Signal_3_LowVol"].map(_state_text)
    transitions["Signal_4_PositiveAR1"] = transitions["Signal_4_PositiveAR1"].map(_state_text)
    transitions["Composite_State"] = transitions["Composite_State"].map(_state_text)
    transitions["Transition_Date"] = transitions.index

    if only_inflection_flips:
        current_state = state_band.loc[change_mask].astype(bool)
        previous_state = state_band.shift().loc[change_mask].astype(bool)
        transitions["Transition_Type"] = [
            "OFF -> ON" if current and not previous else "ON -> OFF"
            for previous, current in zip(previous_state, current_state)
        ]
    else:
        transitions["Transition_Type"] = transitions["Changed_Signals"]

    output_columns = [
        "Transition_Date",
        "Ticker",
        "Daily_Leverage",
        "Analyzed_Close",
        "Signal_1_LongTrend",
        "Signal_2_MediumTrend",
        "Signal_3_LowVol",
        "Signal_4_PositiveAR1",
        "Signal_Count",
        "Composite_State",
        "Transition_Type",
    ]
    return transitions[output_columns]


historical_transitions = historical_signal_transitions(
    ticker=HIST_TICKER,
    daily_leverage=HIST_DAILY_LEVERAGE,
    start_date=HIST_START_DATE,
    end_date=HIST_END_DATE,
    only_inflection_flips=HIST_ONLY_INFLECTION_FLIPS,
)
if historical_transitions.empty:
    print(
        f"Historical transitions for {HIST_TICKER} | Leverage={HIST_DAILY_LEVERAGE:.2f} | Mode={'Inflection flips' if HIST_ONLY_INFLECTION_FLIPS else 'All transitions'} | no transitions in window"
    )
else:
    print(
        f"Historical transitions for {historical_transitions['Ticker'].iloc[0]} | "
        f"Leverage={historical_transitions['Daily_Leverage'].iloc[0]:.2f} | "
        f"Window={historical_transitions['Transition_Date'].min().date()} to {historical_transitions['Transition_Date'].max().date()} | "
        f"Mode={'Inflection flips' if HIST_ONLY_INFLECTION_FLIPS else 'All transitions'}"
    )
display(
    historical_transitions.style.map(
        color_status,
        subset=["Signal_1_LongTrend", "Signal_2_MediumTrend", "Signal_3_LowVol", "Signal_4_PositiveAR1", "Composite_State"],
    )
)


Historical transitions for QQQ | Leverage=1.00 | Window=2000-04-17 to 2025-05-13 | Mode=Inflection flips


,Transition_Date,Ticker,Daily_Leverage,Analyzed_Close,Signal_1_LongTrend,Signal_2_MediumTrend,Signal_3_LowVol,Signal_4_PositiveAR1,Signal_Count,Composite_State,Transition_Type
Date,,,,,,,,,,,
2000-04-17 00:00:00,2000-04-17 00:00:00,QQQ,1.000000,75.521156,ON,OFF,OFF,OFF,1.000000,ON,ON -> OFF
2000-04-18 00:00:00,2000-04-18 00:00:00,QQQ,1.000000,77.522408,ON,OFF,OFF,ON,2.000000,ON,OFF -> ON
2000-04-25 00:00:00,2000-04-25 00:00:00,QQQ,1.000000,76.232124,ON,OFF,OFF,OFF,1.000000,ON,ON -> OFF
2000-07-03 00:00:00,2000-07-03 00:00:00,QQQ,1.000000,80.261017,ON,OFF,ON,OFF,2.000000,ON,OFF -> ON
2000-07-05 00:00:00,2000-07-05 00:00:00,QQQ,1.000000,77.153786,ON,OFF,OFF,OFF,1.000000,ON,ON -> OFF
2000-07-12 00:00:00,2000-07-12 00:00:00,QQQ,1.000000,81.946220,ON,OFF,ON,OFF,2.000000,ON,OFF -> ON
2000-07-28 00:00:00,2000-07-28 00:00:00,QQQ,1.000000,73.203903,ON,OFF,OFF,OFF,1.000000,ON,ON -> OFF
2000-08-18 00:00:00,2000-08-18 00:00:00,QQQ,1.000000,80.261017,ON,OFF,ON,OFF,2.000000,ON,OFF -> ON
2000-09-21 00:00:00,2000-09-21 00:00:00,QQQ,1.000000,73.941208,ON,OFF,OFF,OFF,1.000000,ON,ON -> OFF
